Importing Libaries


In [ ]:
import pandas as pd
import numpy as np
import os
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import warnings
warnings.filterwarnings('ignore')

Loading the datasets


In [ ]:
root_folder = os.getcwd()
dataset_folder = root_folder + "/dataset"
dataframes = []

for filename in os.listdir(dataset_folder):
    file_path = dataset_folder + "/" + filename
    print(file_path)
    try:
        print(f"Opening file {filename}")
        ind_file = pd.ExcelFile(file_path)
        print("------------------------------------------------------")
        sheets_to_process = [sheet for sheet in ind_file.sheet_names if sheet.startswith("Data")]

        for sheet in sheets_to_process:
            df = pd.read_excel(file_path, sheet_name = sheet)
            print(f"  Sheet: {sheets_to_process}, Shape: {df.shape}")
            df = df.iloc[9:].reset_index(drop=True)

            # First column contains the dates
            df_renamed = pd.DataFrame()
            df_renamed['date'] = df.iloc[:, 0]

            # Add metadata columns
            df_renamed['source_file'] = filename
            df_renamed['sheet_name'] = sheet

            
            # Parse date column
            df_renamed['date'] = pd.to_datetime(df_renamed['date'], errors='coerce')
            
            # Extract year from date
            df_renamed['year'] = df_renamed['date'].dt.year
            df_renamed['quarter'] = df_renamed['date'].dt.quarter
            
            # Extract all the measurement data
            for i, col in enumerate(df.columns[1:]):
                # Clean column name - extract measure and state
                col_clean = str(col).strip()
                
                # Parse the column header format: "Measure ; State ;"
                parts = [p.strip().strip(';') for p in col_clean.split(';') if p.strip()]
                
                if len(parts) >= 2:
                    measure = parts[0]
                    state = parts[1]
                else:
                    measure = col_clean
                    state = "Unknown"
                
                # Create a new column with the measurement
                col_name = f"{measure}_{state}".lower().replace(" ", "_")
                try:
                    df_renamed[col_name] = pd.to_numeric(df.iloc[:, i+1], errors='coerce')
                except:
                    pass
            
            dataframes.append(df_renamed)

            print(f"Finished retreiving data from {sheet}")
        
        print("------------------------------------------------------")
        print(f"closing file {filename}")
    except Exception as e:
        print(e)
        print('Above mentioned error has occured')

print(f"\nLoaded {len(dataframes)} sheets successfully!")

COMBINE DATA

In [ ]:
print("\nCombining data from all sheets...")

# Combine all dataframes (vertically stacking)
final_df = pd.concat(dataframes, ignore_index=True)

# Remove rows with missing dates
final_df = final_df.dropna(subset=['date', 'year'])

print(f"Combined data shape: {final_df.shape}")
print(f"Date range: {final_df['date'].min()} to {final_df['date'].max()}")



Combining data from all files...
Combined data shape: (411, 42)

Date range: 1971-06-01 00:00:00 to 2025-09-01 00:00:00


In [ ]:
print(f"\nData loaded from:")
for filename in os.listdir(dataset_folder):
    file_data = final_df[final_df['source_file'] == filename]
    sheets = file_data['sheet_name'].unique() if 'sheet_name' in file_data.columns else ['Data1']
    print(f"  {filename}: {len(file_data)} rows from sheets {list(sheets)}")


Data loaded from:
  dataset/Components of population change - states and territories.xlsx: 178 rows from sheets ['Data1']
  dataset/Population - New South Wales.xlsx: 55 rows from sheets ['Data1']
  dataset/Population - states and territories.xlsx: 178 rows from sheets ['Data1']


In [ ]:
print(f"\nTotal columns available: {len(final_df.columns)}")
print(f"First 5 rows:")
print(final_df.head())


Total columns available: 42
First 5 rows:


,date,year,quarter,source_file,natural_increase_new_south_wales,net_overseas_migration_new_south_wales,net_interstate_migration_new_south_wales,change_over_previous_quarter_new_south_wales,natural_increase_victoria,net_overseas_migration_victoria,...,natural_increase_australian_capital_territory,net_overseas_migration_australian_capital_territory,net_interstate_migration_australian_capital_territory,change_over_previous_quarter_australian_capital_territory,natural_increase_australia,net_overseas_migration_australia,change_over_previous_quarter_australia,estimated_resident_population_male,estimated_resident_population_female,estimated_resident_population_persons
0,1981-06-01,1981,2,dataset/Components of population change - stat...,11516.0,9281.0,-6330.0,NaN,7302.0,5533.0,...,847.0,-131.0,565.0,NaN,33604.0,25575.0,NaN,NaN,NaN,NaN
1,1981-09-01,1981,3,dataset/Components of population change - stat...,9559.0,13284.0,-7768.0,14566.0,7353.0,8237.0,...,777.0,152.0,-15.0,1201.0,29580.0,34141.0,65417.0,NaN,NaN,NaN
2,1981-12-01,1981,4,dataset/Components of population change - stat...,9181.0,13295.0,-4528.0,17439.0,7191.0,8505.0,...,856.0,383.0,-824.0,702.0,29262.0,34482.0,65440.0,NaN,NaN,NaN
3,1982-03-01,1982,1,dataset/Components of population change - stat...,11081.0,12477.0,-3824.0,19225.0,8492.0,7606.0,...,842.0,419.0,-42.0,1506.0,34509.0,31376.0,67581.0,NaN,NaN,NaN
4,1982-06-01,1982,2,dataset/Components of population change - stat...,11097.0,10337.0,-3464.0,17461.0,7310.0,6796.0,...,786.0,271.0,711.0,2055.0,32735.0,28118.0,62549.0,NaN,NaN,NaN


In [ ]:
print("Preparing data for regression analysis...")

# Find columns with natural increase and migration data
natural_increase_cols = [col for col in final_df.columns if 'natural_increase' in col]
net_migration_cols = [col for col in final_df.columns if 'net_overseas_migration' in col]
net_interstate_cols = [col for col in final_df.columns if 'net_interstate_migration' in col]

print(f"Found {len(natural_increase_cols)} natural increase columns")
print(f"Found {len(net_migration_cols)} net overseas migration columns")
print(f"Found {len(net_interstate_cols)} net interstate migration columns")

Preparing data for regression analysis...
Found 9 natural increase columns
Found 9 net overseas migration columns
Found 8 net interstate migration columns


In [ ]:
# Create aggregated columns (Australia-wide)
model_data = pd.DataFrame({
    'date': final_df['date'],
    'year': final_df['year'],
    'quarter': final_df['quarter']
})

# Calculate Australia-wide averages
model_data['natural_increase_au'] = final_df[natural_increase_cols].mean(axis=1)
model_data['net_overseas_migration_au'] = final_df[net_migration_cols].mean(axis=1)
model_data['net_interstate_migration_au'] = final_df[net_interstate_cols].mean(axis=1)

# Remove rows with missing values
model_data = model_data.dropna()

In [ ]:
print(f"\nModel data shape: {model_data.shape}")
print(f"\nFirst few rows:")
print(model_data.head(10))

if len(model_data) > 10 and not model_data[['natural_increase_au', 'net_overseas_migration_au']].isnull().all().all():
    # Prepare features and target
    X = model_data[['net_overseas_migration_au', 'net_interstate_migration_au']]
    y = model_data['natural_increase_au']
    
    # Train/test split
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    # Train model
    model = LinearRegression()
    model.fit(X_train, y_train)
    
    # Make predictions
    y_pred = model.predict(X_test)
    
    # Calculate metrics
    r2 = r2_score(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    
    print("="*60)
    print("ABS DATA REGRESSION ANALYSIS")
    print("="*60)
    print(f"\nModel Performance:")
    print(f"  R2 Score: {r2:.4f}")
    print(f"  RMSE: {rmse:.2f}")
    print(f"  MAE: {mae:.2f}")
    
    print(f"\nModel Coefficients:")
    for col, coef in zip(X.columns, model.coef_):
        print(f"  {col}: {coef:.4f}")
    print(f"  Intercept: {model.intercept_:.4f}")
    
else:
    print("Insufficient data for regression analysis.")

Model data shape: (178, 6)


,date,year,quarter,natural_increase_au,net_overseas_migration_au,net_interstate_migration_au
0,1981-06-01,1981,2,7467.555556,5617.666667,0.0
1,1981-09-01,1981,3,6573.333333,7586.888889,0.0
2,1981-12-01,1981,4,6502.666667,7662.666667,0.0
3,1982-03-01,1982,1,7668.666667,6972.444444,0.0
4,1982-06-01,1982,2,7274.444444,6248.444444,0.0
5,1982-09-01,1982,3,5859.333333,5976.222222,0.0
6,1982-12-01,1982,4,7004.666667,3626.888889,0.0
7,1983-03-01,1983,1,8081.333333,4269.777778,0.0
8,1983-06-01,1983,2,7687.111111,2414.888889,0.0
9,1983-09-01,1983,3,6688.444444,3032.000000,0.0


Visualization D1: Residuals vs Fitted

In [1]:
plt.figure(figsize=(10, 6))
plt.scatter(pred1, residuals1, alpha=0.6)
plt.axhline(y=0, color='r', linestyle='--')
plt.xlabel('Fitted Values')
plt.ylabel('Residuals')
plt.title('Residuals vs Fitted Values - Model 1')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

NameError: name 'plt' is not defined

Visualization D2: Q-Q Plot

In [ ]:
plt.figure(figsize=(10, 6))
stats.probplot(residuals2, dist="norm", plot=plt)
plt.title('Q-Q Plot - Model 2 Residuals')
plt.tight_layout()
plt.show()